# Week 7 ? Feature importance and error analysis

Investigate which inputs the Week 6 forest uses, whether feature removal helps on April, and where predictions fail. Licensed results are generated locally. May is not evaluated here.

## 1. Protocol

December?March trains; April is the repeatedly used validation month. Reuse the frozen Week 6 forest, training-derived price bounds, official district snapshot and preprocessing. Compare six removals with the same rows, seed and forest settings. April selects the largest R?; the reference wins exact ties. Any gain needs confirmation on later unseen months.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
OUT = ROOT / 'outputs' / 'week07'

## 2. Generate and verify local artifacts

The full run takes tens of minutes. It loads the authorized MLS data and saved Week 6 model. Set `RUN_ANALYSIS = True` to regenerate; otherwise inspect existing completed artifacts. Do not upload executed outputs, private predictions or fitted models.

In [ ]:
RUN_ANALYSIS = False
if RUN_ANALYSIS:
    from src.week7 import run
    run(ROOT)
manifest_path = OUT / 'verification.json'
if not manifest_path.exists():
    raise FileNotFoundError('Run scripts/run_week7.py first, or set RUN_ANALYSIS = True.')
import subprocess
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'verify_week7.py')], check=True, cwd=ROOT)
manifest = json.loads(manifest_path.read_text())
assert manifest['may_evaluated'] is False
display(pd.DataFrame(manifest['reference_scores']).T)

## 3. Controlled feature removal

Every removal refits preprocessing and the forest on training only. Removing raw properties retains engineered descendants; removing all geography includes coordinates, ZIP, county, districts and coordinate interactions. Compare dollar and percentage errors alongside R?. These experiments measure predictive substitution, not causality.

In [ ]:
ablation = pd.read_csv(OUT / 'feature_ablation.csv')
display(ablation[['experiment', 'validation_R2', 'validation_MAE_USD', 'validation_MdAPE_pct', 'delta_R2_vs_reference']])
print('April-selected experiment:', manifest['selected'])
display(Image(filename=str(OUT / 'ablation_comparison.png')))

## 4. Individual and grouped permutation importance

Full bounded April data, five deterministic shuffles. Importance is the R? drop after shuffling. A shared row order preserves associations within each group. Correlated inputs can substitute for one another, groups overlap and cannot be summed. Shuffle SD is not a confidence interval; even grouped permutations can create unrealistic cross-group combinations. Impurity importance is a supplementary, potentially biased comparison.

In [ ]:
for filename in ['permutation_importance.csv', 'grouped_permutation_importance.csv']:
    display(pd.read_csv(OUT / filename))
for filename in ['top20_permutation.png', 'grouped_permutation.png', 'impurity_importance.png']:
    display(Image(filename=str(OUT / filename)))

## 5. Prediction errors and geographic coverage

These diagnostics describe the saved Week 6 reference, rather than the selected ablation. Residual = predicted minus actual; positive means overprediction. Price bands use actual prices for retrospective diagnosis only. District/county tables suppress groups below 50 sales; coverage tables require two. Narrow-group R? is unstable. Seen/unseen raw district IDs compare April with training memberships, not encoder buckets.

In [ ]:
display(pd.read_csv(OUT / 'price_band_metrics.csv'))
for column in ['SchoolDistrictElementary', 'SchoolDistrictSecondary', 'SchoolDistrictUnified', 'PostalCode']:
    print(column)
    display(pd.read_csv(OUT / f'coverage_{column}.csv'))
for filename in ['error_diagnostics.png', 'price_band_errors.png', 'district_errors.png']:
    display(Image(filename=str(OUT / filename)))
print('All-cleaned April sensitivity:', manifest['april_all_cleaned'])

## 6. Findings and limitations

Read the generated local report below. Target-bounded scores exclude tails using a price unknown at prediction time. Repeated April selection does not establish future performance. Retrospective district boundaries and sold-snapshot attributes cannot prove historical availability. Confirm with rolling time splits and geographic holdouts before deployment. All licensed results remain local.

In [ ]:
display(Markdown((ROOT / 'docs' / 'WEEK7_RESULTS.md').read_text(encoding='utf-8')))